In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [2]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [3]:
def transform_nus_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [4]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [5]:
with open('../Dataset/Singapore_smsCorpus_en_2015.03.09_all.json', 'r') as file:
        singapore_dataset = json.load(file)
nus_dataset = []
for i in singapore_dataset['smsCorpus']['message']:
    # print(type(i['text']['$']))
    nus_dataset.append(i['text']['$'])

nus_dataset = pd.DataFrame({'Message': nus_dataset})
nus_dataset

,Message
0,Bugis oso near wat...
1,"Go until jurong point, crazy.. Available only ..."
2,I dunno until when... Lets go learn pilates...
3,Den only weekdays got special price... Haiz......
4,Meet after lunch la...
...,...
55830,I LOVE YOU TOO
55831,C-YA
55832,:-)
55833,BE MY GUEST


In [6]:
# nus_dataset = transform_nus_dict(nus_dataset)
# nus_dataset.head()

In [7]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'nus Dataset_'+'.csv')['URL'].to_list())

In [8]:
nus_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'NUS Dataset_'+'.csv')['URL']
nus_dataset['Message Len'] = [len(str(i)) for i in nus_dataset['Message']]
nus_dataset.head()

,Message,Extracted URL,Message Len
0,Bugis oso near wat...,NaN,21
1,"Go until jurong point, crazy.. Available only ...",NaN,111
2,I dunno until when... Lets go learn pilates...,NaN,46
3,Den only weekdays got special price... Haiz......,NaN,140
4,Meet after lunch la...,NaN,22


In [9]:
nus_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'NUS Websites Analysis'+'.csv')
# nus_website_analysis_data = nus_website_analysis_data.drop(columns=['ham', 'spam'])
nus_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,www.hafeezcentre.pkmere,www.hafeezcentre.pkmere.,0,0,-1,0
1,www.united,www.united.,0,0,-1,0
2,www.fiveyearpk.com,www.fiveyearpk.com,0,0,-1,0
3,www.comp.nus.edu.sg/~howyijue/folder,www.comp.nus.edu.sg,328488,10729,200,0
4,www.miworld.com.sg,www.miworld.com.sg,0,0,-1,0


In [10]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [11]:
nus_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23344\2139425198.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  nus_website_analysis_data.iloc[0][0]


'www.hafeezcentre.pkmere'

In [12]:
# for row in nus_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = nus_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(nus_website_analysis_data['FQDN'])}
website_data = nus_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
nus_dataset['FQDN'] = fqdn
nus_dataset['Website Size in KB'] = website_size
nus_dataset['Website Textual Content Length'] = text_content_len
nus_dataset['Status Code'] = status_code
nus_dataset['Parked'] = parked

In [16]:
nus_dataset = nus_dataset.replace('', np.nan)
nus_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23344\1760594351.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  nus_dataset = nus_dataset.replace('', np.nan)


,Message,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,Bugis oso near wat...,NaN,21,NaN,NaN,NaN,NaN,NaN
1,"Go until jurong point, crazy.. Available only ...",NaN,111,NaN,NaN,NaN,NaN,NaN
2,I dunno until when... Lets go learn pilates...,NaN,46,NaN,NaN,NaN,NaN,NaN
3,Den only weekdays got special price... Haiz......,NaN,140,NaN,NaN,NaN,NaN,NaN
4,Meet after lunch la...,NaN,22,NaN,NaN,NaN,NaN,NaN


In [17]:
# Counter(nus_dataset['FQDN'].to_list())
nus_dataset[(nus_dataset['Extracted URL'].notna()) & (nus_dataset['FQDN'].isna())]

,Message,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
# Counter(nus_dataset['FQDN'].to_list())
nus_dataset[(nus_dataset['Extracted URL'].notna()) & (nus_dataset['FQDN'].notna())]

,Message,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
1243,Yup. I dont know how 2 explain.... Need someth...,www.comp.nus.edu.sg/~howyijue/folder,266,www.comp.nus.edu.sg,328488.0,10729.0,200.0,0.0
4508,www.nus.edu.sg then click on students. What ti...,www.nus.edu.sg,161,www.nus.edu.sg,934.0,83.0,200.0,0.0
5137,the url for the game is at www.materiamagica.com,www.materiamagica.com,48,www.materiamagica.com,124535.0,2097.0,200.0,0.0
9602,s new Missed Call Alert service till 30 June. ...,www.miworld.com.sg,74,www.miworld.com.sg,0.0,0.0,-1.0,0.0
10567,HURRAY! www.FullOnSms.com is here! - Save ur m...,www.FullOnSms.com,155,www.FullOnSms.com,1178.0,0.0,200.0,1.0
11570,Is site pe jana n waha pe links h0gi uspe jaka...,www.united,88,www.united.,0.0,0.0,-1.0,0.0
12135,Downld frm www.fiveyearpk.com,www.fiveyearpk.com,29,www.fiveyearpk.com,0.0,0.0,-1.0,0.0
13231,Walk-InJobs Software Engineers recruits at Per...,www.prepareinterview.com,159,www.prepareinterview.com,352470.0,26627.0,200.0,0.0
13853,PaisaPay Check your PaisaPay.in emails every 3...,www.PaisaPay.in,154,www.PaisaPay.in,2308.0,104.0,200.0,0.0
17234,www.ipmart-forum.com,www.ipmart-forum.com,20,www.ipmart-forum.com,482.0,0.0,200.0,1.0


In [19]:
print(len(nus_dataset))

55835


In [20]:
#messages with URL
print(len(nus_dataset[(nus_dataset['Extracted URL'].notna())]), len(nus_dataset[(nus_dataset['Extracted URL'].notna())])/len(nus_dataset))

25 0.0004477478284230321


In [21]:
# #smish messages with URL
# len(nus_dataset[(nus_dataset['Extracted URL'].notna()) & (nus_dataset['class']==1)])

In [22]:
# #smish messages with URL
# len(nus_dataset[(nus_dataset['Extracted URL'].notna()) & (nus_dataset['class']==0)])

In [23]:
#unique FQDN
len(set(nus_dataset[(nus_dataset['FQDN'].notna())]['FQDN']))

22

In [24]:
only_unique_live_websites_data = nus_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

10


In [25]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

3


In [26]:
nus_dataset.to_csv('../Dataset/Refined_NUS_Corpus_Dataset.csv', index=None)